# Romania 2001: acquire the origin-gymnasium reports

This notebook performs **acquisition only**. It fetches the 168 school-specific candidate reports from the public archived Ministry pages and saves **every successful report before moving to the next school**.

The complete source pages and parsed tables contain student names. They are saved only under `~/Desktop/VECTOR_temp/romania_alba_2001_differentiation_v1/`, the temporary folder Charles designated outside the repository. This notebook creates no permanent student-level research file and makes no sorting or placement calculation.

Select the existing `/opt/anaconda3/envs/sports_net/bin/python` interpreter in Cursor. All cells are heavily commented so the process remains inspectable. The live switch starts as `False`; a first **Run All** is a safe preview.

## 1. Choose the run settings

The switch is the only setting you need to change when you are ready. The program checks the Desktop cache first. If 60 reports are already saved, it requests only the remaining 108. A valid zero-student report is still saved as a complete report.

A few connection failures are queued for another pass. Four consecutive failures stop the notebook so a Mac network outage cannot quietly churn all night. Pausing or restarting never deletes completed school files.

In [ ]:
# SAFETY SWITCH. Leave False to inspect; set True when you want to fetch.
RUN_LIVE_ACQUISITION = False

# Starting request spacing. The archive client adapts to success and failure,
# honors any published robots rules, and follows Retry-After when supplied.
INITIAL_INTERVAL_SECONDS = 2.0
MINIMUM_INTERVAL_SECONDS = 1.2
MAXIMUM_INTERVAL_SECONDS = 60.0

# Hard stop for one notebook invocation. Existing school files survive.
MAX_RUNTIME_HOURS = 6.0

# A later pass revisits transiently refused addresses. You can also simply
# rerun this notebook tomorrow; completed files are verified and skipped.
RETRY_PASSES = 3
RETRY_COOLDOWN_SECONDS = 180.0


## 2. Load the acquisition program

This is a local import. It does **not** make an archive request. The printout confirms which Python environment Cursor selected and where the private Desktop cache lives.

In [ ]:
# Use the existing sports_net Python interpreter in Cursor.
# This cell finds the education workspace whether Cursor starts the kernel
# beside this notebook or at the repository root. It does not use the network.
from pathlib import Path
import importlib
import json
import sys

def find_education_root():
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        for candidate in (
            parent,
            parent / "education_analysis",
            parent / "3-Master_Plan" / "VECTOR_work" / "new_VECTOR_work" / "education_analysis",
        ):
            if (candidate / "code" / "EDUCATION_20260930_romania_cache_workflow.py").is_file():
                return candidate.resolve()
    raise FileNotFoundError("Could not find the education_analysis workspace from this notebook's working directory.")

EDUCATION_ROOT = find_education_root()
CODE_DIR = EDUCATION_ROOT / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import EDUCATION_20260930_romania_cache_workflow as workflow
print("Python interpreter:", sys.executable)
print("Education workspace:", EDUCATION_ROOT)
print("Desktop source cache:", workflow.CACHE)


## 3. Inspect saved progress

This cell reads the structural audit and the current Desktop cache. It verifies the identity, row count, and digests of any saved report. It prints only school counts and source codes, never student names.

In [ ]:
status = workflow.cache_status()
print(f"Verified school reports saved: {status['cached_and_verified_codes']}/{status['directory_codes']}")
print(f"Reports still needed: {status['remaining_codes']}")
print("Temporary source folder:", status['cache_folder'])
if status['remaining_codes']:
    print("First ten remaining source codes:", status['remaining_source_codes'][:10])


## 4. Acquire or resume

Set `RUN_LIVE_ACQUISITION = True` above, then run this cell. The progress bar shows the pass and total reports saved. Every ten schools, and after any failed request, a plain-language checkpoint gives the code, status, and rough time remaining.

A successful school is written as a single atomic JSON file. That file contains both the original archived HTML and the parsed source rows. The run verifies its digest immediately. On interruption, rerun this cell; verified files are skipped. If the archive becomes unavailable, stop and resume later.

When all 168 files pass verification, the program writes `completed_manifest.json` in the Desktop cache. The analysis notebook requires that manifest.

In [ ]:
if not RUN_LIVE_ACQUISITION:
    print("PREVIEW ONLY: no archive requests were made.")
    print("Set RUN_LIVE_ACQUISITION = True in Section 1 to begin or resume.")
else:
    try:
        status = workflow.acquire(
            live=True,
            initial_interval_seconds=INITIAL_INTERVAL_SECONDS,
            minimum_interval_seconds=MINIMUM_INTERVAL_SECONDS,
            maximum_interval_seconds=MAXIMUM_INTERVAL_SECONDS,
            max_runtime_hours=MAX_RUNTIME_HOURS,
            retry_passes=RETRY_PASSES,
            retry_cooldown_seconds=RETRY_COOLDOWN_SECONDS,
        )
    except (KeyboardInterrupt, workflow.load_structural_module().RetrievalStopped) as error:
        # An interruption leaves every previously finished school file intact.
        print("Retrieval stopped:", str(error) or "interrupted by user")
    finally:
        status = workflow.cache_status()
        print(f"Saved and verified: {status['cached_and_verified_codes']}/{status['directory_codes']}")
        print(f"Still needed: {status['remaining_codes']}")


## 5. Completion check

The acquisition stage is complete only if this cell reports **168 saved, zero remaining**, and `completed_manifest.json` exists. The separate analysis notebook then works entirely from these saved pages, without contacting Wayback.

In [ ]:
status = workflow.cache_status()
manifest_path = workflow.CACHE / "completed_manifest.json"
print(f"Saved: {status['cached_and_verified_codes']}/{status['directory_codes']}")
print("Complete manifest present:", manifest_path.exists())
if status['remaining_codes'] == 0 and manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("Acquisition complete at:", manifest['completed_utc'])
else:
    print("Acquisition remains resumable. Re-run Section 4 when ready.")
